# 🔡 Traductor Neuronal Shiwilu ↔ Español a Nivel de Caracteres
**Objetivo:** Entrenar un modelo de traducción automática neuronal (NMT) a nivel de caracteres con corpus paralelo Shiwilu–Español.

Este enfoque es útil para lenguas de escasos recursos donde no existen tokenizadores adecuados o vocabularios estandarizados.

🧪 Tecnologías: `Transformers`, `datasets`, entrenamiento desde cero (sin preentrenamiento).


## 1️⃣ Preparación del entorno

In [ ]:
!pip install -q transformers datasets sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.3 MB/s eta 0:00:00


## 2️⃣ Carga del corpus y conversión a caracteres

In [ ]:
from datasets import Dataset

# Leer archivos
with open("shiwilu.txt", encoding='utf-8') as f:
    shiwilu = [list(line.strip()) for line in f.readlines()]

with open("espanol.txt", encoding='utf-8') as f:
    espanol = [list(line.strip()) for line in f.readlines()]

examples = [{"shw": ''.join(s), "es": ''.join(e)} for s, e in zip(shiwilu, espanol)]
data = Dataset.from_list(examples).train_test_split(test_size=0.1)
data

DatasetDict({
    train: Dataset({
        features: ['shw', 'es'],
        num_rows: 346
    })
    test: Dataset({
        features: ['shw', 'es'],
        num_rows: 39
    })
})

## 3️⃣ Tokenización a nivel de caracteres

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Lista de textos (shiwilu + español)
with open("shiwilu.txt", encoding="utf-8") as f1, open("espanol.txt", encoding="utf-8") as f2:
    lines = f1.readlines() + f2.readlines()

# Entrenamiento tokenizer carácter por carácter
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=200,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

tokenizer.train_from_iterator(lines, trainer)

# Guardar tokenizer
tokenizer.save("char_tokenizer.json")


In [ ]:
from transformers import PreTrainedTokenizerFast

chars = sorted(set(''.join([''.join(x) for x in shiwilu + espanol])))
vocab = {c: i+4 for i, c in enumerate(chars)}  # Reservar ids: 0-pad, 1-unk, 2-bos, 3-eos
vocab["[PAD]"] = 0
vocab["[UNK]"] = 1
vocab["[BOS]"] = 2
vocab["[EOS]"] = 3

tokenizer = PreTrainedTokenizerFast(tokenizer_file="char_tokenizer.json")
tokenizer.add_tokens(list(vocab.keys()))
tokenizer.pad_token = "[PAD]"
tokenizer.unk_token = "[UNK]"
tokenizer.bos_token = "[BOS]"
tokenizer.eos_token = "[EOS]"



## 4️⃣ Preparación para el entrenamiento

In [ ]:
from transformers import EncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments

# Modelo encoder-decoder base (desde cero)
model = EncoderDecoderModel.from_encoder_decoder_pretrained("bert-base-uncased", "bert-base-uncased")
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
# model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.encoder.resize_token_embeddings(len(tokenizer))
model.decoder.resize_token_embeddings(len(tokenizer))

def preprocess(example):
    inputs = tokenizer(example['shw'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    targets = tokenizer(example['es'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_data = data.map(preprocess, batched=True, remove_columns=["shw", "es"])


Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

Map:   0%|          | 0/346 [00:00<?, ? examples/s]

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

## 5️⃣ Entrenamiento del modelo

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./char_nmt_shiwilu",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=30,
    weight_decay=0.01,
    predict_with_generate=True,
    save_total_limit=2,
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"]
)

In [ ]:
trainer.train()  # Descomenta para entrenar

/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:631: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:651: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Step,Training Loss
500,0.824400
1000,0.576200


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:631: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:651: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/usr/local/lib/python3.11/dist-packages/tra

TrainOutput(global_step=1320, training_loss=0.6533815094918916, metrics={'train_runtime': 703.2005, 'train_samples_per_second': 14.761, 'train_steps_per_second': 1.877, 'total_flos': 1591671676584960.0, 'train_loss': 0.6533815094918916, 'epoch': 30.0})

## 6️⃣ Traducción de prueba

In [ ]:
import torch

# Detecta si hay GPU disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mueve el modelo al dispositivo
model.to(device)

# Mueve los tensores al mismo dispositivo
inputs = tokenizer("NAWA'INPAMU ITEKLALLINA", return_tensors="pt", padding=True, truncation=True, max_length=128)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

# Genera traducción
output_ids = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    decoder_start_token_id=tokenizer.bos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_length=128
)

# Decodifica salida
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


e l l o s   o o a a a                           a a .


In [ ]:
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt", padding=True, truncation=True, max_length=128)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_start_token_id=tokenizer.bos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_length=128
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

## 📏 Evaluación del Modelo con BLEU y Ejemplos de Traducción

In [ ]:
from sacrebleu import corpus_bleu

refs = []
hyps = []

for example in data["test"]:
    source = example["shw"]
    reference = example["es"]
    generated = traducir(source)  # 👈 Usamos función definida antes
    refs.append([reference])
    hyps.append(generated)

# Calcular BLEU
bleu = corpus_bleu(hyps, list(zip(*refs)))
print(f"BLEU score: {bleu.score:.2f}")


BLEU score: 0.17


### 🔍 Ejemplos de traducción generados por el modelo

In [ ]:
# Mostrar algunos ejemplos de traducción del conjunto de prueba
for i in range(5):
    entrada = data['test'][i]['shw']        # oración en shiwilu
    referencia = data['test'][i]['es']      # traducción esperada en español
    traduccion = traducir(entrada)             # traducción generada por el modelo
    print(f"🔸 Entrada:     {entrada}")
    print(f"🎯 Referencia: {referencia}")
    print(f"🤖 Traducción: {traduccion}\n")


🔸 Entrada:     KUDA KU'LA A'LEKTAPI'ÑIDEK ALA'SA' LLINSERPI
🎯 Referencia: nosotras no enseñamos una escritura.
🤖 Traducción: l a   n ñ ñ o             e e e a                         a a a .

🔸 Entrada:     NANA WILLAKU'APER KU'LA LUNPI'ÑI ÑINCHITUNAN'PIDEK'KEK
🎯 Referencia: la niña no habla en la escuela.
🤖 Traducción: l a s   n i ñ o o                 r e r   a                 a a a .

🔸 Entrada:     NANA WILA KUAPER KU'LA CHINTEK'IÑI MUTUPIK
🎯 Referencia: la niña no sube a la montaña.
🤖 Traducción: l a   n i ñ a         e e e a a                       a a .

🔸 Entrada:     NAWA'INPAMU ITEKLALLINA
🎯 Referencia: ellos se lavan las manos.
🤖 Traducción: e l l o s   e o a a a                         a a .

🔸 Entrada:     KENMA KU'LA LUNCHI'NA ALA'SA' KERKA'
🎯 Referencia: tú no lees un libro.
🤖 Traducción: e l l a s   n o   e e e e a a a                       a a .

